In [18]:
import os

In [19]:
!pip install -q youtube-transcript-api langchain-community langchain langchain-openai langchain-google-genai \
               faiss-cpu tiktoken python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [ ]:
video_id = "ru44DngJYoA "

try:
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id)
    transcript = " ".join(
        chunk.text for chunk in transcript_list
    )

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

You take a deep breath and then continue speaking. The reason people aren't comfortable with the pause is because they don't know what the pause is for. Right? The pause allows people to process what you're saying. Think about it now. Listeners, as you're listening to that, the moment I paused, you had a moment to process the things that I was saying. Do you want to be more charismatic? You see, in the world of communication, there are two fundamental areas. Didn't that just seem really important? I was so bad at interactions with human beings. There was just a period of my life where I I didn't understand what anybody was saying to me. You're saying that everyone can go from being a shy, unconfident speaker to being a prolific speaker like yourself. Being a confident communicator, that's just another series of behaviors that you can practice. So when people say, "Oh, I'm shy." I always say to them, "Oh, that's because you've been practicing the shy behaviors for the last 15, 20, 30, 4

In [22]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='You take a deep breath and then continue', start=0.08, duration=4.64), FetchedTranscriptSnippet(text="speaking. The reason people aren't", start=2.96, duration=3.04), FetchedTranscriptSnippet(text='comfortable with the pause is because', start=4.72, duration=2.8), FetchedTranscriptSnippet(text="they don't know what the pause is for.", start=6.0, duration=3.679), FetchedTranscriptSnippet(text='Right? The pause allows people to', start=7.52, duration=3.6), FetchedTranscriptSnippet(text="process what you're saying. Think about", start=9.679, duration=2.88), FetchedTranscriptSnippet(text="it now. Listeners, as you're listening", start=11.12, duration=2.76), FetchedTranscriptSnippet(text='to that, the moment I', start=12.559, duration=3.601), FetchedTranscriptSnippet(text='paused, you had a moment to process the', start=13.88, duration=3.76), FetchedTranscriptSnippet(text='things that I was', start=16.16, duration=3.68), FetchedTran

## Step 1b - Indexing (Text Splitting)

In [23]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [24]:
len(chunks)

149

In [25]:
chunks[0]

Document(metadata={}, page_content='You take a deep breath and then continue speaking. The reason people aren\'t comfortable with the pause is because they don\'t know what the pause is for. Right? The pause allows people to process what you\'re saying. Think about it now. Listeners, as you\'re listening to that, the moment I paused, you had a moment to process the things that I was saying. Do you want to be more charismatic? You see, in the world of communication, there are two fundamental areas. Didn\'t that just seem really important? I was so bad at interactions with human beings. There was just a period of my life where I I didn\'t understand what anybody was saying to me. You\'re saying that everyone can go from being a shy, unconfident speaker to being a prolific speaker like yourself. Being a confident communicator, that\'s just another series of behaviors that you can practice. So when people say, "Oh, I\'m shy." I always say to them, "Oh, that\'s because you\'ve been practici

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)